# 21 — Bài báo, bước 2c: PhoBERT và flat Bi-LSTM (Kaggle, GPU)

**Phạm vi**

PhoBERT (`vinai/phobert-base-v2`, chiến lược E) cho tầng 1, tầng 2 và phẳng bốn lớp, cùng Bi-LSTM + fastText phẳng bốn lớp.
Thiết kế, lưới và tiêu chí chọn đã pre-register ở amendment v3.4, giống hệt quy trình của ViSoBERT và Bi-LSTM ở GĐ6, GĐ7, GĐ9.

**Dữ liệu trên Kaggle**

Dataset gồm `shopee_train.csv`, `shopee_validation.csv` và ba file text-only `holdout_text_only.csv`, `focused_text_only.csv`,
`lazada_text_only.csv`. Không được có file nhãn của tập khóa; notebook kiểm tra điều này trước khi chạy.

**Yêu cầu trước khi chạy**

Bật GPU và Internet, có secret `GITHUB_TOKEN`. Code của bước 2c (`02_src/src/classification/strategy_phobert/`) phải đã được
push lên GitHub, vì notebook clone repo để import.

**Đầu ra**

Thư mục `/kaggle/working/paper_step2c/`: lưới, bảng Validation, quét ngưỡng, chi phí, cấu hình đã chọn và xác suất dự đoán
của Validation cùng ba tập khóa. Notebook không chấm điểm ba tập khóa.

In [ ]:
import os, subprocess, sys, json, gc, datetime, time
from kaggle_secrets import UserSecretsClient
REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
REPO_DIR = '/kaggle/working/repo'
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
assert os.path.exists(os.path.join(REPO_DIR, '02_src', 'src', 'classification', 'strategy_phobert', 'train.py')), \
    'Repo trên GitHub chưa có strategy_phobert — cần push code bước 2c trước khi chạy.'
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py_vncorenlp', 'underthesea'], capture_output=True, text=True)
print('pip install py_vncorenlp underthesea:', 'ok' if r.returncode == 0 else r.stderr[-800:])
sys.path.append(os.path.join(REPO_DIR, '02_src'))
import numpy as np, pandas as pd, torch, transformers, yaml
from src.utils.seeds import set_seed
set_seed(42)
COMMIT = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
try:
    JAVA = subprocess.run(['java', '-version'], capture_output=True, text=True).stderr.splitlines()[0]
except FileNotFoundError:
    JAVA = 'không có Java'
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print('Ngày chạy:', datetime.date.today().isoformat(), '| commit:', COMMIT)
print('transformers', transformers.__version__, '| torch', torch.__version__, '| GPU:', GPU, '| Java:', JAVA)

## Cấu hình nền và lưới

**PhoBERT**

Cấu hình nền lấy từ cấu hình ViSoBERT đã khóa của từng tầng (max_len 256, batch 16, warmup 10%, weight decay 0,01, class_weight
balanced), chỉ đổi tên mô hình. Lưới: tầng 1 và phẳng lr {1e-5, 2e-5, 3e-5} × epoch {3, 4, 5}; tầng 2 lr {1e-5, 2e-5, 3e-5} ×
epoch {4, 6, 8}.

**Flat Bi-LSTM**

Cấu hình nền là Bi-LSTM tầng 1 đã khóa (embedding fastText cố định, 10 epoch, max_len 128, batch 32); lưới hidden {64, 128, 256}
× dropout {0,3; 0,5} × learning rate {1e-3; 2e-3} như tầng 1.

In [ ]:
from src.classification.final_eval import load_frozen, LOCKED, TEXT_ONLY, predictions_long
from src.classification.strategy_phobert.train import MODEL_NAME
FZ = load_frozen(REPO_DIR)
SEEDS = FZ['tier1']['seeds']
assert SEEDS == [42, 1337, 2024]
PB_BASE = {t: {**FZ[t]['visobert']['config'], 'model_name': MODEL_NAME} for t in ('tier1', 'tier2', 'flat')}
LR = (1e-5, 2e-5, 3e-5)
PB_GRID = {'tier1': [{'lr': lr, 'epochs': e} for lr in LR for e in (3, 4, 5)],
           'tier2': [{'lr': lr, 'epochs': e} for lr in LR for e in (4, 6, 8)],
           'flat': [{'lr': lr, 'epochs': e} for lr in LR for e in (3, 4, 5)]}
BCFG = FZ['tier1']['bilstm']['config']
BL_GRID = [{'hidden_size': h, 'dropout': d, 'lr': lr} for h in (64, 128, 256) for d in (0.3, 0.5) for lr in (1e-3, 2e-3)]
for t, c in PB_BASE.items():
    print(f'PhoBERT {t}:', {k: c[k] for k in ('max_len', 'batch_size', 'warmup_ratio', 'weight_decay', 'class_weight')}, '|', len(PB_GRID[t]), 'cấu hình')
print('Flat Bi-LSTM:', {k: BCFG[k] for k in ('epochs', 'max_len', 'batch_size', 'class_weight', 'freeze_embedding')}, '|', len(BL_GRID), 'cấu hình')

## Dữ liệu

**Tập có nhãn**

Train 1.200 và Validation 350.

**Tập khóa (chỉ văn bản)**

Ba file text-only phải chỉ có `review_id` và `text`, không trùng review với Train hay Validation. Nếu dataset chứa file có nhãn
của tập khóa, notebook dừng.

In [ ]:
def find_file(name, root='/kaggle/input'):
    hits = [os.path.join(d, name) for d, _, fs in os.walk(root) if name in fs]
    if not hits:
        listing = [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs]
        raise FileNotFoundError(f'Không tìm thấy {name} trong {root}. Các file hiện có:\n' + ('\n'.join(listing) or '(trống)'))
    return hits[0]

labeled_locked = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs if f in LOCKED.values()]
assert not labeled_locked, f'Phát hiện file CÓ NHÃN của tập khóa trên Kaggle: {labeled_locked}'
train_df = pd.read_csv(find_file('shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(find_file('shopee_validation.csv'), encoding='utf-8-sig')
LOCKED_DF = {name: pd.read_csv(find_file(TEXT_ONLY.format(name)), encoding='utf-8-sig') for name in LOCKED}
seen = set(train_df['review_id']) | set(val_df['review_id'])
for name, df in LOCKED_DF.items():
    assert list(df.columns) == ['review_id', 'text'], f'{name}: file trên Kaggle phải chỉ có review_id và text'
    assert not set(df['review_id']) & seen, f'{name}: trùng review với Train/Validation'
assert (len(train_df), len(val_df)) == (1200, 350)
assert {k: len(v) for k, v in LOCKED_DF.items()} == {'holdout': 1000, 'focused': 150, 'lazada': 500}
print('Train', len(train_df), '| Validation', len(val_df), '|', {k: len(v) for k, v in LOCKED_DF.items()})

## Tách từ và kiểm tra tokenizer PhoBERT

**Bộ tách từ**

RDRSegmenter của VnCoreNLP; nếu không khởi động được thì dùng underthesea và ghi lại. Thời gian tách từ đo trên 1.000 review
Hold-out text-only khi bộ nhớ đệm còn trống, rồi mọi tập được tách sẵn một lần.

**Kiểm tra tokenizer**

Tỷ lệ token `<unk>` trên Train và phép thử MLM che từng token trên 60 review Train; top-1 phải trên 30%, như kiểm tra ViSoBERT ở GĐ6.

In [ ]:
from src.classification.strategy_phobert.segment import Segmenter
from src.classification.strategy_phobert.train import load_tokenizer, unk_rate, mlm_alignment, PhobertRunner
SEG = Segmenter('auto', save_dir='/kaggle/working/vncorenlp')
print('Bộ tách từ:', SEG.backend, '| lỗi khởi tạo:', SEG.errors or 'không')
print('Ví dụ:', SEG(['thủ tục hủy hàng rắc rối', 'app lag quá, thanh toán không được']))
t0 = time.time()
SEG(LOCKED_DF['holdout']['text'].astype(str).tolist())
SEG_S_PER_1000 = (time.time() - t0) / len(LOCKED_DF['holdout']) * 1000
for df in (train_df, val_df, *LOCKED_DF.values()):
    SEG(df['text'].astype(str).tolist())
print(f'Tách từ: {SEG_S_PER_1000:.2f} giây / 1.000 review | review tách lỗi (giữ nguyên): {SEG.failures}')
TOK = load_tokenizer(MODEL_NAME)
seg_train = SEG(train_df['text'].astype(str).tolist())
UNK, MEAN_TOK = unk_rate(TOK, seg_train)
ALIGN = mlm_alignment(TOK, MODEL_NAME, pd.Series(seg_train).sample(60, random_state=42).tolist())
print(f"Tokenizer {type(TOK).__name__}: <unk> {UNK:.3%} | {MEAN_TOK:.1f} token/review | MLM top-1 {ALIGN['mlm_top1']:.1%} trên {ALIGN['n_tokens']} token")
assert ALIGN['mlm_top1'] > 0.3, 'Tokenizer PhoBERT không khớp embedding — dừng, không train.'
gc.collect(); torch.cuda.empty_cache()

## Dò lưới và chạy ba seed

**Quy trình**

Mỗi cấu hình train một lần với seed 42 và được xếp theo Macro-F1 trên Validation ở argmax (hai lớp cho tầng 1, ba lớp trên
gold-actionable cho tầng 2, bốn lớp cho phẳng). Cấu hình thắng chạy ba seed, mỗi lần dự đoán Validation và ba tập khóa.

**Dữ liệu từng tầng**

Tầng 1 và phẳng train trên Train 1.200 gồm ambiguous; tầng 2 train trên Train actionable bỏ ambiguous và dự đoán cả review
không actionable để ghép cascade sau này.

In [ ]:
from src.classification.rq1 import grid_search, run_multi, BilstmRunner
from src.classification.learning_curve import tier1_metrics, tier2_metrics
from src.classification import evaluate as E
from src.classification.tier1_actionable.mapping import to_tier1, TIER1_LABELS
from src.classification.tier2_bfu.mapping import to_tier2, TIER2_LABELS
from src.classification.flat_baseline.mapping import to_flat, FLAT_LABELS
DATA_T = {'tier1': (to_tier1(train_df), to_tier1(val_df), TIER1_LABELS, lambda y, p: tier1_metrics(np.asarray(y), np.asarray(p).argmax(1))),
          'tier2': (to_tier2(train_df, drop_ambiguous=True), to_tier2(val_df, drop_ambiguous=False), TIER2_LABELS,
                    lambda y, p: tier2_metrics(np.asarray(y), np.asarray(p).argmax(1))),
          'flat': (to_flat(train_df), to_flat(val_df), FLAT_LABELS, lambda y, p: E.metrics(np.asarray(y), np.asarray(p).argmax(1), FLAT_LABELS))}
SETS = {'validation': val_df, **LOCKED_DF}
GRIDS, BEST, RUNS = {}, {}, {}

def tune_and_run(tier, model, runner, grid):
    tr, va, names, mfn = DATA_T[tier]
    g, best = grid_search(runner, grid, tr, va, len(names), names, mfn, strategy=f'{model} {tier}')
    GRIDS[(tier, model)], BEST[(tier, model)] = g, best
    print(f'{model} {tier}: Macro-F1 Validation {g["macro_f1"].min():.3f}–{g["macro_f1"].max():.3f} | thắng {best}')
    for s in SEEDS:
        r = RUNS[(tier, model, s)] = run_multi(runner, tr, SETS, len(names), names, s, best)
        print(f"  seed {s}: train {r['train_seconds']:.0f}s | suy luận {r['infer_seconds_per_1000']:.2f}s/1.000 | GPU {r['peak_gpu_gb']} GB")

for tier in ('tier1', 'tier2', 'flat'):
    tune_and_run(tier, 'phobert', PhobertRunner(PB_BASE[tier], SEG), PB_GRID[tier])

In [ ]:
from src.classification.strategy_bilstm_fasttext.embeddings import build_vocab, download, load_vectors
vec = download(BCFG['fasttext_url'], '/tmp/cc.vi.300.vec.gz')
vocab = build_vocab(pd.concat([train_df['text'], val_df['text']]).astype(str))
emb, coverage = load_vectors(vec, vocab, dim=int(BCFG['embedding_dim']))
print(f'Từ vựng {len(vocab):,} | có vector fastText: {coverage:.1%}')
tune_and_run('flat', 'bilstm', BilstmRunner(BCFG, vocab, emb), BL_GRID)

## Ngưỡng tầng 1 và bảng Validation

**Ngưỡng PhoBERT tầng 1**

Luật v2.0 trên xác suất ACTIONABLE của seed 42: trong các ngưỡng 0,01–0,90 có recall ACTIONABLE ≥ 0,80, lấy ngưỡng có Macro-F1
cao nhất.

**Bảng so sánh**

Tầng 1 ở ngưỡng đã chọn, tầng 2 trên gold-actionable, cascade PhoBERT và hai mô hình phẳng, đặt cạnh số Validation đã có của
ViSoBERT và Bi-LSTM. Validation chỉ có 71 review actionable nên chênh lệch nhỏ chưa có ý nghĩa.

In [ ]:
from src.classification.tier1_actionable import threshold as TH
y1 = to_tier1(val_df)['y'].to_numpy()
v2 = to_tier2(val_df, drop_ambiguous=False)
mask2 = val_df['review_id'].isin(v2['review_id']).to_numpy()
y2, y4 = v2['y'].to_numpy(), to_flat(val_df)['y'].to_numpy()
pv = lambda tier, model, s: RUNS[(tier, model, s)]['probs']['validation']
SW1 = TH.sweep(y1, pv('tier1', 'phobert', 42)[:, 1])
THR1 = TH.choose(SW1)
REC_MIN = min(TH.metrics_at(y1, pv('tier1', 'phobert', s)[:, 1], THR1['threshold'])['recall_actionable'] for s in SEEDS)
print('Ngưỡng PhoBERT tầng 1:', THR1, '| recall thấp nhất qua ba seed:', round(REC_MIN, 3))
ROW1 = TH.summary_row(y1, [{'probs': pv('tier1', 'phobert', s)} for s in SEEDS], THR1, 'PhoBERT', BEST[('tier1', 'phobert')])
ROW2 = E.summary_row(y2, [{'probs': pv('tier2', 'phobert', s)[mask2]} for s in SEEDS], 'PhoBERT', BEST[('tier2', 'phobert')],
                     'Validation (gold-actionable)', TIER2_LABELS)
CAS = [E.cascade_labels(pv('tier1', 'phobert', s)[:, 1] >= THR1['threshold'], pv('tier2', 'phobert', s).argmax(1)) for s in SEEDS]
ROWC = E.summary_row(y4, [{'pred': p} for p in CAS], 'Cascade PhoBERT', f"tầng 1 ngưỡng {THR1['threshold']} + tầng 2", 'Validation', FLAT_LABELS)
ROWFP = E.summary_row(y4, [{'probs': pv('flat', 'phobert', s)} for s in SEEDS], 'Phẳng PhoBERT', BEST[('flat', 'phobert')], 'Validation', FLAT_LABELS)
ROWFB = E.summary_row(y4, [{'probs': pv('flat', 'bilstm', s)} for s in SEEDS], 'Phẳng Bi-LSTM + fastText', BEST[('flat', 'bilstm')], 'Validation', FLAT_LABELS)
RES = os.path.join(REPO_DIR, '05_experiments', 'rq1_classification')
GATE = pd.read_csv(os.path.join(RES, 'gd10_gate_validation.csv')).set_index('model')
CVF = pd.read_csv(os.path.join(RES, 'cascade_vs_flat_validation.csv')).dropna(subset=['tập']).set_index('strategy')
g = lambda k: f"{GATE.loc[k, 'new_mean']:.1f} ± {GATE.loc[k, 'new_std']:.1f}"
VAL = pd.DataFrame([
    {'chỉ số': 'Tầng 1 Macro-F1 (2 lớp, ở ngưỡng)', 'PhoBERT': ROW1['macro_f1'], 'ViSoBERT': g('Tầng 1 – ViSoBERT (ngưỡng 0.11)'),
     'Bi-LSTM + fastText': g('Tầng 1 – Bi-LSTM + fastText (ngưỡng 0.09)')},
    {'chỉ số': 'Tầng 2 Macro-F1 (3 lớp)', 'PhoBERT': ROW2['macro_f1'], 'ViSoBERT': g('Tầng 2 – ViSoBERT'), 'Bi-LSTM + fastText': g('Tầng 2 – Bi-LSTM + fastText')},
    {'chỉ số': 'End-to-end cascade Macro-F1 (4 lớp)', 'PhoBERT': ROWC['macro_f1'], 'ViSoBERT': CVF.loc['Cascade ViSoBERT', 'macro_f1'], 'Bi-LSTM + fastText': ''},
    {'chỉ số': 'End-to-end phẳng Macro-F1 (4 lớp)', 'PhoBERT': ROWFP['macro_f1'], 'ViSoBERT': CVF.loc['Baseline phẳng ViSoBERT', 'macro_f1'],
     'Bi-LSTM + fastText': ROWFB['macro_f1']}])
print(VAL.to_string(index=False))
print(); print(pd.DataFrame([ROW2, ROWC, ROWFP, ROWFB]).to_string(index=False))

## Lưu kết quả

**Các file**

Lưới, bảng Validation, quét ngưỡng, chi phí (cùng cột với `gd10_costs_neural.csv`, thêm thời gian tách từ của PhoBERT), cấu hình
đã chọn `paper_neural_config.yaml` và xác suất dự đoán `paper_pred_neural_{validation, holdout, focused, lazada}.csv`.

**Mang về máy**

Tải thư mục `paper_step2c/` về `05_experiments/rq1_classification/`; chép `paper_neural_config.yaml` vào `03_configs/frozen/`.

In [ ]:
from src.utils.kaggle_paths import working_dir
OUT = working_dir('paper_step2c')
save = lambda df, name: df.to_csv(os.path.join(OUT, f'{name}.csv'), index=False, encoding='utf-8-sig')
save(pd.concat([g.assign(tier=t, model=m) for (t, m), g in GRIDS.items()], ignore_index=True), 'paper_neural_grid')
save(pd.DataFrame([ROW1]), 'paper_neural_validation_tier1')
save(pd.DataFrame([ROW2, ROWC, ROWFP, ROWFB]), 'paper_neural_validation')
save(VAL, 'paper_neural_validation_compare')
save(SW1, 'paper_phobert_tier1_threshold_sweep')
cost = []
for tier, model in BEST:
    rs = [RUNS[(tier, model, s)] for s in SEEDS]
    cost.append({'tier': tier, 'model': model, 'n_train': rs[0]['n_train'],
                 'train_seconds_mean': float(np.mean([r['train_seconds'] for r in rs])),
                 'infer_seconds_per_1000_mean': float(np.mean([r['infer_seconds_per_1000'] for r in rs])),
                 'peak_gpu_gb_max': max((r['peak_gpu_gb'] or 0) for r in rs), 'gpu': GPU,
                 'segment_seconds_per_1000': SEG_S_PER_1000 if model == 'phobert' else 0.0})
save(pd.DataFrame(cost), 'paper_neural_costs')
save(predictions_long(val_df['review_id'], RUNS, 'validation'), 'paper_pred_neural_validation')
for name, df in LOCKED_DF.items():
    save(predictions_long(df['review_id'], RUNS, name), f'paper_pred_neural_{name}')
TODAY = datetime.date.today().strftime('%d/%m/%Y')
CONTENT = {'paper_neural': {
    'amendment': 'v3.4', 'seeds': SEEDS, 'canonical_seed': 42,
    'phobert': {'model_name': MODEL_NAME, 'segmenter': SEG.backend, 'segmenter_errors': SEG.errors, 'segment_failures': SEG.failures,
                'tokenizer_check': {'unk_rate': round(float(UNK), 5), 'mean_tokens': round(float(MEAN_TOK), 2),
                                    'mlm_top1': round(float(ALIGN['mlm_top1']), 4)},
                'tier1': {'config': {**PB_BASE['tier1'], **BEST[('tier1', 'phobert')]}, 'threshold': THR1['threshold'],
                          'threshold_rule': {'min_recall': 0.8, 'sweep': [0.01, 0.9, 0.01], 'chosen_on_seed': 42, 'amendment': 'v2.0'},
                          'recall_constraint_met': bool(THR1['recall_constraint_met']), 'recall_min_over_seeds_validation': round(float(REC_MIN), 3)},
                'tier2': {'config': {**PB_BASE['tier2'], **BEST[('tier2', 'phobert')]}, 'decision': 'argmax'},
                'flat': {'config': {**PB_BASE['flat'], **BEST[('flat', 'phobert')]}, 'decision': 'argmax'}},
    'bilstm_flat': {'config': {**BCFG, **BEST[('flat', 'bilstm')]}, 'decision': 'argmax', 'fasttext_coverage': round(float(coverage), 4)},
    'environment': {'commit': str(COMMIT), 'gpu': str(GPU), 'transformers': str(transformers.__version__), 'torch': str(torch.__version__),
                    'java': str(JAVA)}}}
head = (f'# FROZEN — khóa ngày {TODAY}, amendment v3.4 (chọn trên Validation trong notebook 21, Kaggle). '
        'Không sửa sau mốc này; mọi thay đổi phải có amendment mới.\n')
open(os.path.join(OUT, 'paper_neural_config.yaml'), 'w', encoding='utf-8').write(head + yaml.safe_dump(CONTENT, allow_unicode=True, sort_keys=False))
json.dump({'run_date': datetime.date.today().isoformat(), 'commit': COMMIT, 'gpu': GPU, 'segmenter': SEG.backend},
          open(os.path.join(OUT, 'paper_step2c_run_info.json'), 'w'), indent=1)
print('Đã lưu:', sorted(os.listdir(OUT)))